# ICBHI Respiratory Sound Dataset Exploration

This notebook is the initial exploration of the ICBHI respiratory sound data used for the respiratory-sound ML pipeline. The focus is on understanding the recordings, annotations, respiratory-cycle structure, and spectro-temporal representations before model development.

**Task:** classify respiratory cycles as normal, crackle, wheeze, or crackle+wheeze.

**Exploration goals**
- inspect the recording and annotation structure;
- verify the annotation format and cycle timing;
- visualize representative respiratory signals;
- compare basic time-frequency representations; and
- use these observations to define the downstream preprocessing contract.

In [ ]:
from pathlib import Path
data_dir= Path("../data/raw")
wav_files = list(data_dir.rglob("*.wav"))
print("Number of Wav Files:",len(wav_files))
for file in wav_files[:5]:
 print(file)


In [ ]:
txt_files = list(data_dir.rglob("*.txt"))
print("Number of Txt Files:",len(txt_files))


In [ ]:
print("Number of Wav Files:",len(wav_files))
print("Number of Txt Files:",len(txt_files))


In [ ]:
audio_path = wav_files[0]
annotation_path = audio_path.with_suffix(".txt")
print("audio:",audio_path)
print("annotation:",annotation_path)
print("annotation path exists:",annotation_path.exists())


## 1. Load and inspect an annotated recording


In [ ]:
import pandas as pd
annotation = pd.read_csv(annotation_path, sep="\t", header=None)
annotation.columns = ["start","end","crackle","wheeze"]
annotation.head()


In [ ]:
print("Number of annotated cycles:",len(annotation))
print("\nColumn  names:")
for col in annotation.columns:
    print(f" - {col}")
print("\nFirst row:")
print(annotation.iloc[0])


## 2. Inspect the raw respiratory waveform


In [ ]:
import librosa
from numpy import number
audio_path = file
audio, sr = librosa.load(audio_path, sr=None)
print("sampling rate:",sr)
print("number of samples:",len(audio))
print("duration:",len(audio)/sr,"seconds")


In [ ]:
import matplotlib.pyplot as plt
import librosa.display
plt.figure(figsize=(14, 4))
librosa.display.waveshow(audio, sr=sr)
plt.title("ICBHI respiratory recording")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.show()


## 3. Inspect annotated respiratory cycles


In [ ]:
import pandas as pd
annotation_path = audio_path.with_suffix(".txt")
annotations = pd.read_csv(annotation_path, sep="\t", header=None)
annotations.columns = ["start","end","crackle","wheeze"]
annotations.head()
print("number of annotated cycles:",len(annotations))
print("\nColumn names:")
print(annotations.columns)
print("\nFirst row:")
print(annotations.iloc[0])


In [ ]:
start_time =annotations.iloc[0]["start"]
end_time = annotations.iloc[0]["end"]
start_sample = int(start_time * sr)
end_sample = int(end_time * sr) 
segment = audio[start_sample:end_sample]
print("start time:", start_time)
print("end time:", end_time)
print("start sample:", start_sample)
print("end sample:", end_sample)
print("segment length:", len(segment))
print("segment duration:", len(segment)/sr, "seconds")


In [ ]:
import matplotlib.pyplot as plt
import librosa.display
plt.figure(figsize=(12, 4))
librosa.display.waveplot(segment, sr=sr)
plt.title(f"annotated segment


## 4. Build a fixed-length spectrogram representation


In [ ]:
import librosa
import numpy as np

# Audio configuration parameters
SAMPLE_RATE = 16000
TARGET_DURATION = 2.0  # seconds
TARGET_SAMPLES = int(SAMPLE_RATE * TARGET_DURATION)  # 32,000 samples

# Load WAV file
wav_path = "../data/raw/101_1b1_Al_sc_Meditron.wav"
y, sr = librosa.load(wav_path, sr=SAMPLE_RATE)

# Extract timing for cycle 0
start_sec = annotation.iloc[0]['start']
end_sec = annotation.iloc[0]['end']

# Convert seconds to sample indices
start_idx = int(start_sec * sr)
end_idx = int(end_sec * sr)

# Crop raw audio slice
pcm_slice = y[start_idx:end_idx]

# Apply length standardization (Zero Padding or Truncation)
if len(pcm_slice) < TARGET_SAMPLES:
    padding_needed = TARGET_SAMPLES - len(pcm_slice)
    pcm_fixed = np.pad(pcm_slice, (0, padding_needed), mode='constant')
else:
    pcm_fixed = pcm_slice[:TARGET_SAMPLES]

print(f"Original Cycle Duration: {end_sec - start_sec:.3f} seconds ({len(pcm_slice)} samples)")
print(f"Standardized Buffer Shape: {pcm_fixed.shape} samples")


In [ ]:
# Compute Mel Spectrogram
mel_spec = librosa.feature.melspectrogram(
    y=pcm_fixed,
    sr=SAMPLE_RATE,
    n_fft=1024,
    hop_length=512,
    n_mels=64
)

# Convert power scale to Decibels (Log Scale)
log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)

# Add Channel Dimension for 2D CNN input format: (Height, Width, Channels)
feature_tensor = log_mel_spec[..., np.newaxis]

print(f"Log-Mel Spectrogram Shape: {log_mel_spec.shape}")  # Should be (64, 63)
print(f"Model Input Tensor Shape:  {feature_tensor.shape}")  # Should be (64, 63, 1)


In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
import librosa.display

# Target dimensions for fixed-size 2D matrix
TARGET_TIME_FRAMES = 63  # Equivalent to 2.0s at sr=16000, hop=512
# Compute Mel Spectrogram ONLY on active audio cycle (NO audio zero-padding beforehand)
mel_spec_active = librosa.feature.melspectrogram(
    y=pcm_slice,
    sr=SAMPLE_RATE,
    n_fft=1024,
    hop_length=512,
    n_mels=64
)
# Convert active cycle to dB scale
log_mel_active = librosa.power_to_db(mel_spec_active, ref=np.max, top_db=80)
# Normalize the ACTIVE sound features to [0.1, 1.0] 
# (Reserving 0.0 strictly for matrix zero-padding)
min_val = log_mel_active.min()
max_val = log_mel_active.max()
log_mel_norm_active = 0.1 + 0.9 * (log_mel_active - min_val) / (max_val - min_val + 1e-6)
# Matrix-level Zero Padding along the Time Axis (Width)
current_frames = log_mel_norm_active.shape[1]

if current_frames < TARGET_TIME_FRAMES:
    pad_width = TARGET_TIME_FRAMES - current_frames
    # Pad right side with 0.0 (background/silence)
    log_mel_padded = np.pad(log_mel_norm_active, ((0, 0), (0, pad_width)), mode='constant', constant_values=0.0)
else:
    log_mel_padded = log_mel_norm_active[:, :TARGET_TIME_FRAMES]
# Add Channel Dimension for 2D CNN input format
feature_tensor = log_mel_padded[..., np.newaxis]

print(f"Padded Feature Tensor Shape: {feature_tensor.shape}")  # (64, 63, 1)
# Plot the clean, correctly scaled spectrogram
plt.figure(figsize=(8, 4))
librosa.display.specshow(
    log_mel_padded, 
    sr=SAMPLE_RATE, 
    hop_length=512, 
    x_axis='time', 
    y_axis='mel'
)
plt.colorbar(label="Normalized Feature Value")
plt.title(f"Cycle 0 Correctly Scaled Spectrogram (Crackle: {annotation.iloc[0]['crackle']}, Wheeze: {annotation.iloc[0]['wheeze']})")
plt.tight_layout()
plt.savefig("cycle_0_correct_scaling.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Print summary of the feature tensor
print(f"Feature Tensor Shape: {feature_tensor.shape}")  # Expecting (64, 63, 1)
print(f"Min Value (Padding):  {feature_tensor.min():.1f}")  # Should be 0.0
print(f"Max Value (Peak):     {feature_tensor.max():.1f}")  # Should be 1.0

# Print distribution of non-zero vs zero-padded cells
non_zero_ratio = (feature_tensor > 0.0).sum() / feature_tensor.size
print(f"Active Feature Content: {non_zero_ratio * 100:.1f}%")
print(f"Zero-Padded Silence:   {(1 - non_zero_ratio) * 100:.1f}%")


## 5. Compare spectro-temporal representations


In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

# Set up figure for comparisons
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
# Standard STFT Log-Spectrogram
stft = np.abs(librosa.stft(pcm_slice, n_fft=1024, hop_length=512))
stft_db = librosa.amplitude_to_db(stft, ref=np.max)
librosa.display.specshow(stft_db, sr=SAMPLE_RATE, hop_length=512, x_axis='time', y_axis='log', ax=axes[0, 0])
axes[0, 0].set_title("1. Standard STFT (Log Hz)")
# MFCCs (20 coefficients)
mfccs = librosa.feature.mfcc(y=pcm_slice, sr=SAMPLE_RATE, n_mfcc=20, n_fft=1024, hop_length=512)
librosa.display.specshow(mfccs, sr=SAMPLE_RATE, hop_length=512, x_axis='time', ax=axes[0, 1])
axes[0, 1].set_title("2. MFCCs (20 Coefficients)")
# Continuous Wavelet Transform (CWT Scalogram)
# Note: Requires PyWavelets (pip install PyWavelets)
try:
    import pywt
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "PyWavelets"])
    import pywt
widths = np.arange(1, 65)
cwt_matrix, _ = pywt.cwt(pcm_slice, widths, 'morl')
cwt_abs = np.abs(cwt_matrix)
axes[1, 0].imshow(cwt_abs, aspect='auto', cmap='magma', origin='lower')
axes[1, 0].set_title("3. CWT Scalogram (Morlet Wavelet)")
axes[1, 0].set_ylabel("Scale")
# Chromagram
chroma = librosa.feature.chroma_stft(y=pcm_slice, sr=SAMPLE_RATE, n_fft=1024, hop_length=512)
librosa.display.specshow(chroma, sr=SAMPLE_RATE, hop_length=512, x_axis='time', y_axis='chroma', ax=axes[1, 1])
axes[1, 1].set_title("4. Chromagram")

plt.tight_layout()
plt.savefig("alternative_features_comparison.png", dpi=300)
plt.show()


## 6. Inspect crackle-containing examples


In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import pywt
# Find the first row in your annotations DataFrame that contains a crackle
crackle_rows = annotation[annotation["crackle"] == 1].copy()

if len(crackle_rows) == 0:
    print("No crackle cycles found in the current annotation file.")
else:
    # Pick the first crackle cycle
    crackle_cycle_idx = crackle_rows.index[0]
    row = annotation.loc[crackle_cycle_idx]
    pcm_data = y

    start_sec = row['start']
    end_sec = row['end']
    
    print(f"Found Crackle at Cycle Index {crackle_cycle_idx}")
    print(f"Time Range: {start_sec:.3f}s to {end_sec:.3f}s | Crackle: {row['crackle']} | Wheeze: {row['wheeze']}")
# Slice the audio corresponding to this crackle cycle
    start_sample = int(start_sec * SAMPLE_RATE)
    end_sample = int(end_sec * SAMPLE_RATE)
    pcm_crackle_slice = pcm_data[start_sample:end_sample]
# Generate Log-Mel Spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y=pcm_crackle_slice,
        sr=SAMPLE_RATE,
        n_fft=1024,
        hop_length=512,
        n_mels=64
    )
    log_mel = librosa.power_to_db(mel_spec, ref=np.max, top_db=80)
# Generate Continuous Wavelet Transform (CWT Scalogram)
    widths = np.arange(1, 65)
    cwt_matrix, _ = pywt.cwt(pcm_crackle_slice, widths, 'morl')
    cwt_abs = np.abs(cwt_matrix)
# Plot Comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot Log-Mel Spectrogram
    img1 = librosa.display.specshow(
        log_mel, 
        sr=SAMPLE_RATE, 
        hop_length=512, 
        x_axis='time', 
        y_axis='mel', 
        ax=axes[0]
    )
    axes[0].set_title(f"Cycle {crackle_cycle_idx} Log-Mel Spectrogram (Crackle=1)")
    fig.colorbar(img1, ax=axes[0], format="%+2.0f dB")

    # Plot CWT Scalogram
    img2 = axes[1].imshow(cwt_abs, aspect='auto', cmap='magma', origin='lower')
    axes[1].set_title(f"Cycle {crackle_cycle_idx} CWT Scalogram (Morlet Wavelet)")
    axes[1].set_xlabel("Samples")
    axes[1].set_ylabel("Scale (Frequency)")
    fig.colorbar(img2, ax=axes[1])

    plt.tight_layout()
    plt.savefig("crackle_cycle_visuals.png", dpi=300)
    plt.show()


In [ ]:
from pathlib import Path
import pandas as pd

# Path to your raw dataset directory
raw_dir = Path("../data/raw") # Adjust relative path if needed

crackle_file = None

# Iterate through all annotation .txt files
for txt_path in sorted(raw_dir.glob("*.txt")):
    # Read annotation file (columns: start, end, crackle, wheeze)
    df_temp = pd.read_csv(txt_path, sep='\t', header=None, names=['start', 'end', 'crackle', 'wheeze'])
    
    if (df_temp['crackle'] == 1).any():
        crackle_file = txt_path
        print(f"Found crackles in file: {txt_path.name}")
        break

if crackle_file:
# Load the corresponding audio file (.wav)
    wav_path = crackle_file.with_suffix('.wav')
    pcm_data, SAMPLE_RATE = librosa.load(wav_path, sr=16000)
# Load the annotations for this specific file
    df_annot = pd.read_csv(crackle_file, sep='\t', header=None, names=['start', 'end', 'crackle', 'wheeze'])
# Filter for the first crackle cycle
    crackle_cycle_idx = df_annot[df_annot['crackle'] == 1].index[0]
    row = df_annot.loc[crackle_cycle_idx]
    
    print(f"\nProcessing Cycle {crackle_cycle_idx}:")
    print(f"Time Range: {row['start']:.3f}s to {row['end']:.3f}s | Crackle: {row['crackle']} | Wheeze: {row['wheeze']}")
else:
    print("No crackle annotations found in data/raw.")


In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import pywt
# Update the file name directly to the target crackle file
wav_path = raw_dir / "104_1b1_Ll_sc_Litt3200.wav"
pcm_data, sr = librosa.load(wav_path, sr=16000)
# Set the exact time range for Cycle 4 (8.829s to 10.396s)
start_sample = int(8.829 * sr)
end_sample = int(10.396 * sr)
pcm_crackle_cycle = pcm_data[start_sample:end_sample]
# Log-Mel Spectrogram
mel_spec = librosa.feature.melspectrogram(
    y=pcm_crackle_cycle,
    sr=sr,
    n_fft=1024,
    hop_length=512,
    n_mels=64
)
log_mel = librosa.power_to_db(mel_spec, ref=np.max, top_db=80)
# CWT Scalogram
widths = np.arange(1, 65)
cwt_matrix, _ = pywt.cwt(pcm_crackle_cycle, widths, 'morl')
cwt_abs = np.abs(cwt_matrix)
# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

librosa.display.specshow(log_mel, sr=sr, hop_length=512, x_axis='time', y_axis='mel', ax=axes[0])
axes[0].set_title("104_1b1 Cycle 4: Log-Mel Spectrogram (Crackle=1)")

axes[1].imshow(cwt_abs, aspect='auto', cmap='magma', origin='lower')
axes[1].set_title("104_1b1 Cycle 4: CWT Scalogram (Morlet)")
axes[1].set_xlabel("Samples")

plt.tight_layout()
plt.show()


In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import pywt
# Target settings (2.0 seconds at 16kHz)
SAMPLE_RATE = 16000
TARGET_DURATION = 2.0
TARGET_SAMPLES = int(SAMPLE_RATE * TARGET_DURATION)  # 32,000 samples
TARGET_TIME_FRAMES = 63                                # 63 Mel frames
# Extract active cycle (104_1b1 Cycle 4: 8.829s to 10.396s)
start_sample = int(8.829 * SAMPLE_RATE)
end_sample = int(10.396 * SAMPLE_RATE)
pcm_slice = pcm_data[start_sample:end_sample]

# --- FEATURE 1: Padded Log-Mel Spectrogram ---
mel_spec = librosa.feature.melspectrogram(y=pcm_slice, sr=SAMPLE_RATE, n_fft=1024, hop_length=512, n_mels=64)
log_mel = librosa.power_to_db(mel_spec, ref=np.max, top_db=80)
# Normalize active region [0.1, 1.0]
log_mel_norm = 0.1 + 0.9 * (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-6)
# Pad matrix width to 63 frames
log_mel_padded = np.pad(log_mel_norm, ((0, 0), (0, max(0, TARGET_TIME_FRAMES - log_mel_norm.shape[1]))), mode='constant')[:, :TARGET_TIME_FRAMES]

# --- FEATURE 2: Padded CWT Scalogram ---
# Pad audio waveform to 32,000 samples FIRST so CWT scales match time directly
pcm_padded = np.pad(pcm_slice, (0, max(0, TARGET_SAMPLES - len(pcm_slice))), mode='constant')[:TARGET_SAMPLES]
widths = np.arange(1, 65)
cwt_matrix, _ = pywt.cwt(pcm_padded, widths, 'morl')
cwt_abs = np.abs(cwt_matrix)
# Normalize CWT
cwt_norm = (cwt_abs - cwt_abs.min()) / (cwt_abs.max() - cwt_abs.min() + 1e-6)

# --- PLOT SIDE-BY-SIDE ON THE SAME 2.0s TIME AXIS ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Standardized Log-Mel
img1 = librosa.display.specshow(
    log_mel_padded, sr=SAMPLE_RATE, hop_length=512, x_axis='time', y_axis='mel', ax=axes[0]
)
axes[0].set_xlim([0, 2.0])
axes[0].set_title("1. Standardized Log-Mel (0.0s to 2.0s)")
fig.colorbar(img1, ax=axes[0], label="Normalized Intensity")

# Plot 2: Standardized CWT Scalogram
# Map x-extent to [0, 2.0] seconds instead of raw samples
img2 = axes[1].imshow(
    cwt_norm, aspect='auto', cmap='magma', origin='lower', extent=[0, 2.0, 1, 64]
)
axes[1].set_xlim([0, 2.0])
axes[1].set_title("2. Standardized CWT Scalogram (0.0s to 2.0s)")
axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Scale (Frequency)")
fig.colorbar(img2, ax=axes[1], label="Normalized Intensity")

plt.tight_layout()
plt.savefig("aligned_2sec_comparison.png", dpi=300)
plt.show()


## 7. Frame-level padding experiment


In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

def process_and_pad_cycle(file_path, start_s, end_s, sr=16000, target_frames=63):
    # Load audio segment
    y, _ = librosa.load(file_path, sr=sr)
    pcm_slice = y[int(start_s * sr):int(end_s * sr)]
    
    # Compute Mel Spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y=pcm_slice, sr=sr, n_fft=1024, hop_length=512, n_mels=64
    )
    log_mel = librosa.power_to_db(mel_spec, ref=np.max, top_db=80)
    
    # Active Normalization
    log_mel_norm = 0.1 + 0.9 * (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-6)
    
    # Matrix Pad to 63 time frames
    current_frames = log_mel_norm.shape[1]
    if current_frames < target_frames:
        pad_width = target_frames - current_frames
        log_mel_padded = np.pad(log_mel_norm, ((0, 0), (0, pad_width)), mode='constant', constant_values=0.0)
    else:
        log_mel_padded = log_mel_norm[:, :target_frames]
        
    return log_mel_padded
# Process Normal Cycle (Patient 101, Cycle 0)
normal_spec = process_and_pad_cycle(
    raw_dir / "101_1b1_Al_sc_Meditron.wav", 0.036, 0.579
)
# Process Crackle Cycle (Patient 104, Cycle 4)
crackle_spec = process_and_pad_cycle(
    raw_dir / "104_1b1_Ll_sc_Litt3200.wav", 8.829, 10.396
)
# Side-by-Side Visual Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Normal
img1 = librosa.display.specshow(normal_spec, sr=16000, hop_length=512, x_axis='time', y_axis='mel', ax=axes[0])
axes[0].set_title("NORMAL (Patient 101, Cycle 0)")
axes[0].set_xlim([0, 2.0])
fig.colorbar(img1, ax=axes[0], label="Normalized Intensity")

# Plot Crackle
img2 = librosa.display.specshow(crackle_spec, sr=16000, hop_length=512, x_axis='time', y_axis='mel', ax=axes[1])
axes[1].set_title("CRACKLE (Patient 104, Cycle 4)")
axes[1].set_xlim([0, 2.0])
fig.colorbar(img2, ax=axes[1], label="Normalized Intensity")

plt.tight_layout()
plt.savefig("normal_vs_crackle_side_by_side.png", dpi=300)
plt.show()


## 8. Exploration takeaways

The exploration establishes the basic data contract used in the ML notebook: annotated respiratory-cycle intervals, fixed-rate audio preprocessing, log-Mel spectrogram features, and explicit handling of variable-duration cycles. The later model-development notebook applies these choices to patient-separated training, validation, and evaluation.